In [5]:
#!/usr/bin/env python3
"""
Greedy neuron-addition test accuracy experiment.

Goal
----
For k = 1, 2, ..., n_neurons:

    1. Among all remaining neurons, choose the neuron whose addition gives
       the best TRAIN balanced accuracy.

    2. Add that neuron to the selected set.

    3. Report TRAIN and TEST accuracy/balanced accuracy for the selected set.

Important
---------
The TEST set is never used to select neurons.
It is only used for reporting after each train-selected addition.

This uses a cheap mean-difference decoder:

    w_j = mean_animate_j - mean_inanimate_j
    midpoint_j = 0.5 * (mean_animate_j + mean_inanimate_j)

    contribution_ij = (x_ij - midpoint_j) * w_j

The score for a selected neuron set S is:

    score_i = sum_{j in S} contribution_ij

Prediction:

    animate if score_i > 0

Expected files
--------------
/home/maria/mousehash/analysis_data/
    hybrid_neural_responses.npy
    google_vit-base-patch16-224_embeddings_logits.pkl

Outputs
-------
/home/maria/mousehash/best_neuron_addition_test_accuracy/
    vit_derived_labels.csv
    stimulus_presentation_counts.csv
    kept_neuron_indices.csv
    train_test_split.csv
    greedy_neuron_addition_curve.csv
    summary.json
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split


# =============================================================================
# Config
# =============================================================================
BASE_DIR = Path("/home/maria/Science/thesis/experiments/007--PoorMansClassifier")
DATA_DIR = Path("/home/maria/Science/data")
OUT_DIR = BASE_DIR / "best_neuron_addition_test_accuracy"
OUT_DIR.mkdir(exist_ok=True, parents=True)

NEURAL_FILE = DATA_DIR / "hybrid_neural_responses_reduced.npy"
VIT_FILE = DATA_DIR / "google_vit-base-patch16-224_embeddings_logits.pkl"
VIT_KEY = "natural_scenes"

N_STIMULI = 118
RANDOM_SEED = 42

ANIMATE_TOP1_THRESHOLD = 397

PRESENTATION_ORDER = "block"
STIMULUS_IDS_FILE = DATA_DIR / "stimulus_ids.npy"

# I strongly recommend NOT running all neurons first.
# Start with 500 or 1000, then increase if needed.
MAX_ADDITIONS: int | None = 1000

CANDIDATE_BATCH_SIZE = 2048

# =============================================================================
# Loading
# =============================================================================

def load_neural_presentations() -> np.ndarray:
    """
    Load neural matrix and return shape:

        presentations x neurons

    Expected raw shape is usually:

        neurons x presentations = (39209, 5900)
    """
    if not NEURAL_FILE.exists():
        raise FileNotFoundError(f"Missing neural file: {NEURAL_FILE}")

    X_raw = np.asarray(np.load(NEURAL_FILE, allow_pickle=True))

    print(f"[INFO] Raw neural shape: {X_raw.shape}")

    if X_raw.ndim != 2:
        raise ValueError(f"Expected 2D neural matrix, got {X_raw.shape}")

    n0, n1 = X_raw.shape

    if n0 > n1 and n1 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as neurons x presentations.")
        X_pres = X_raw.T
    elif n1 > n0 and n0 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as presentations x neurons.")
        X_pres = X_raw
    else:
        raise ValueError(
            f"Could not infer orientation from neural shape {X_raw.shape}. "
            "Expected something like (39209, 5900) or (5900, 39209)."
        )

    X_pres = X_pres.astype(np.float32, copy=False)

    print(f"[INFO] Presentation-level neural shape: {X_pres.shape}")

    return X_pres


def load_vit_natural_scenes_logits() -> np.ndarray:
    """
    Load ViT logits for natural scenes.

    Expected:

        np.load(VIT_FILE, allow_pickle=True)["natural_scenes"]

    Expected output shape:

        (118, 1000)
    """
    if not VIT_FILE.exists():
        raise FileNotFoundError(f"Missing ViT file: {VIT_FILE}")

    obj = np.load(VIT_FILE, allow_pickle=True)

    if hasattr(obj, "keys"):
        keys = list(obj.keys())

        if VIT_KEY not in keys:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in {VIT_FILE}. "
                f"Available keys: {keys}"
            )

        logits = np.asarray(obj[VIT_KEY])

    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        item = obj.item()

        if not isinstance(item, dict):
            raise TypeError(f"Expected object array containing dict, got {type(item)}")

        if VIT_KEY not in item:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in object dict. "
                f"Available keys: {list(item.keys())}"
            )

        logits = np.asarray(item[VIT_KEY])

    else:
        raise TypeError(f"Unsupported ViT object type: {type(obj)}")

    if logits.ndim != 2:
        raise ValueError(f"Expected 2D ViT logits, got {logits.shape}")

    if logits.shape[0] != N_STIMULI:
        raise ValueError(
            f"Expected {N_STIMULI} rows, got {logits.shape[0]}"
        )

    print(f"[INFO] ViT logits shape: {logits.shape}")

    return logits.astype(np.float32, copy=False)


def make_labels_from_vit_logits(logits: np.ndarray) -> np.ndarray:
    """
    Derive binary labels.

        1 = animate
        0 = inanimate

    Rule:

        top1 <= 397 => animate
    """
    top1 = np.argmax(logits, axis=1)
    y = (top1 <= ANIMATE_TOP1_THRESHOLD).astype(int)

    print("[INFO] Derived animate/inanimate labels from ViT top-1.")
    print(f"[INFO] Inanimate count: {int((y == 0).sum())}")
    print(f"[INFO] Animate count:   {int((y == 1).sum())}")

    pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "top1_imagenet_class": top1,
            "label_animate": y,
        }
    ).to_csv(OUT_DIR / "vit_derived_labels.csv", index=False)

    return y


# =============================================================================
# Presentation averaging
# =============================================================================

def make_presentation_stimulus_ids(n_presentations: int) -> np.ndarray:
    """
    Return vector of length n_presentations containing stimulus IDs 0..117.
    """
    if STIMULUS_IDS_FILE.exists():
        stim_ids = np.load(STIMULUS_IDS_FILE, allow_pickle=True).astype(int).ravel()

        if len(stim_ids) != n_presentations:
            raise ValueError(
                f"{STIMULUS_IDS_FILE} has length {len(stim_ids)}, "
                f"but neural data has {n_presentations} presentations."
            )

        if stim_ids.min() < 0 or stim_ids.max() >= N_STIMULI:
            raise ValueError(
                f"Stimulus IDs must be in [0, {N_STIMULI - 1}], "
                f"got min={stim_ids.min()}, max={stim_ids.max()}."
            )

        print(f"[INFO] Loaded explicit stimulus IDs from {STIMULUS_IDS_FILE}")
        return stim_ids

    if n_presentations % N_STIMULI != 0:
        raise ValueError(
            f"n_presentations={n_presentations} is not divisible by {N_STIMULI}."
        )

    repeats = n_presentations // N_STIMULI

    if PRESENTATION_ORDER == "block":
        stim_ids = np.repeat(np.arange(N_STIMULI), repeats)
    elif PRESENTATION_ORDER == "cycle":
        stim_ids = np.tile(np.arange(N_STIMULI), repeats)
    else:
        raise ValueError("PRESENTATION_ORDER must be either 'block' or 'cycle'.")

    print(
        f"[WARN] No explicit {STIMULUS_IDS_FILE.name} found. "
        f"Assuming PRESENTATION_ORDER={PRESENTATION_ORDER!r}. "
        f"Repeats per stimulus={repeats}."
    )

    return stim_ids.astype(int)


def average_presentations_by_stimulus(X_pres: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Average presentation-level neural matrix to stimulus-level matrix.

    Input:

        X_pres: presentations x neurons

    Output:

        X_avg: stimuli x neurons
        counts: presentations per stimulus
    """
    n_presentations, n_neurons = X_pres.shape
    stim_ids = make_presentation_stimulus_ids(n_presentations)

    X_avg = np.zeros((N_STIMULI, n_neurons), dtype=np.float32)
    counts = np.zeros(N_STIMULI, dtype=int)

    for stim_id in range(N_STIMULI):
        mask = stim_ids == stim_id
        counts[stim_id] = int(mask.sum())

        if counts[stim_id] == 0:
            raise ValueError(f"Stimulus {stim_id} has zero presentations.")

        X_avg[stim_id] = X_pres[mask].mean(axis=0)

    print("[INFO] Averaged neural responses by stimulus.")
    print(f"[INFO] Stimulus-averaged neural shape: {X_avg.shape}")
    print(f"[INFO] Presentations per stimulus: min={counts.min()}, max={counts.max()}")

    pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "n_presentations": counts,
        }
    ).to_csv(OUT_DIR / "stimulus_presentation_counts.csv", index=False)

    return X_avg, counts


# =============================================================================
# Cleaning and splitting
# =============================================================================

def clean_features_using_train(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Remove neurons that are non-finite anywhere or have zero variance on train.

    Zero-variance should be checked on train because the model only learns from train.
    """
    finite = np.isfinite(X_train).all(axis=0) & np.isfinite(X_test).all(axis=0)
    train_var = np.nanvar(X_train, axis=0)
    nonzero_train_var = train_var > 0

    keep = finite & nonzero_train_var
    kept_original_indices = np.where(keep)[0]

    removed = X_train.shape[1] - int(keep.sum())
    if removed:
        print(f"[WARN] Removing {removed} non-finite or train-zero-variance neurons.")

    X_train_clean = X_train[:, keep].astype(np.float32, copy=False)
    X_test_clean = X_test[:, keep].astype(np.float32, copy=False)

    pd.DataFrame(
        {
            "clean_feature_index": np.arange(len(kept_original_indices)),
            "original_neuron_index": kept_original_indices,
        }
    ).to_csv(OUT_DIR / "kept_neuron_indices.csv", index=False)

    print(f"[INFO] Clean train shape: {X_train_clean.shape}")
    print(f"[INFO] Clean test shape:  {X_test_clean.shape}")

    return X_train_clean, X_test_clean, kept_original_indices


def make_stratified_train_test_split(
    X: np.ndarray,
    y: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Make 80/20 stratified split over the 118 stimulus-level rows.
    """
    indices = np.arange(len(y))

    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    split_name = np.full(len(y), "train", dtype=object)
    split_name[test_idx] = "test"

    split_df = pd.DataFrame(
        {
            "stimulus_index": np.arange(len(y)),
            "label_animate": y,
            "split": split_name,
        }
    )
    split_df.to_csv(OUT_DIR / "train_test_split.csv", index=False)

    print("[INFO] Stratified 80/20 train/test split:")
    for name, idx in [("train", train_idx), ("test", test_idx)]:
        counts = np.bincount(y[idx], minlength=2)
        print(
            f"  {name:5s}: n={len(idx):3d}, "
            f"inanimate={counts[0]:3d}, animate={counts[1]:3d}"
        )

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx], train_idx, test_idx


# =============================================================================
# Cheap decoder
# =============================================================================

def compute_mean_difference_contributions(
    X_fit: np.ndarray,
    y_fit: np.ndarray,
    X_eval: np.ndarray,
) -> np.ndarray:
    """
    Fit mean-difference decoder on X_fit/y_fit and return per-neuron contributions
    on X_eval.

    For each neuron j:

        w_j = mu_animate_j - mu_inanimate_j
        midpoint_j = 0.5 * (mu_animate_j + mu_inanimate_j)

        contribution_ij = (x_ij - midpoint_j) * w_j

    For selected neuron set S:

        score_i = sum_j contribution_ij
    """
    mu0 = X_fit[y_fit == 0].mean(axis=0)
    mu1 = X_fit[y_fit == 1].mean(axis=0)

    w = mu1 - mu0
    midpoint = 0.5 * (mu1 + mu0)

    return ((X_eval - midpoint) * w).astype(np.float32, copy=False)


def scores_to_predictions(scores: np.ndarray) -> np.ndarray:
    return (scores > 0).astype(int)


def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, scores))


def accuracy_stats(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    preds = scores_to_predictions(scores)

    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
        "auc": safe_auc(y_true, scores),
    }


def balanced_accuracy_for_candidate_matrix(
    y_true: np.ndarray,
    score_matrix: np.ndarray,
) -> np.ndarray:
    """
    Fast balanced accuracy for many candidate score vectors.

    score_matrix shape:

        n_samples x n_candidates
    """
    preds = score_matrix > 0
    y_bool = y_true.astype(bool)

    pos_mask = y_bool
    neg_mask = ~y_bool

    if pos_mask.sum() == 0 or neg_mask.sum() == 0:
        raise ValueError("Need both classes to compute balanced accuracy.")

    tpr = preds[pos_mask].mean(axis=0)
    tnr = (~preds[neg_mask]).mean(axis=0)

    return 0.5 * (tpr + tnr)


# =============================================================================
# Greedy selection
# =============================================================================

def find_best_next_neuron_by_train_balanced_accuracy(
    train_contrib: np.ndarray,
    y_train: np.ndarray,
    current_train_scores: np.ndarray,
    remaining_indices: np.ndarray,
    batch_size: int,
) -> tuple[int, float]:
    """
    Among remaining neurons, find the neuron whose addition gives the best
    TRAIN balanced accuracy.
    """
    best_feature_idx = None
    best_score = -np.inf

    for start in range(0, len(remaining_indices), batch_size):
        batch = remaining_indices[start:start + batch_size]

        candidate_score_matrix = current_train_scores[:, None] + train_contrib[:, batch]

        batch_scores = balanced_accuracy_for_candidate_matrix(
            y_train,
            candidate_score_matrix,
        )

        local_best_pos = int(np.argmax(batch_scores))
        local_best_score = float(batch_scores[local_best_pos])

        if local_best_score > best_score:
            best_score = local_best_score
            best_feature_idx = int(batch[local_best_pos])

    assert best_feature_idx is not None

    return best_feature_idx, best_score


def greedy_neuron_addition_curve(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    kept_original_neuron_indices: np.ndarray,
    max_additions: int | None,
) -> pd.DataFrame:
    """
    Greedily add neurons.

    Selection:

        best next neuron = neuron maximizing TRAIN balanced accuracy.

    Reporting:

        train accuracy
        train balanced accuracy
        train AUC
        test accuracy
        test balanced accuracy
        test AUC
    """
    n_train, n_features = X_train.shape

    if max_additions is None:
        max_additions = n_features
    else:
        max_additions = min(max_additions, n_features)

    print("[INFO] Computing train/test contribution matrices...")
    train_contrib = compute_mean_difference_contributions(X_train, y_train, X_train)
    test_contrib = compute_mean_difference_contributions(X_train, y_train, X_test)

    selected: list[int] = []
    remaining_mask = np.ones(n_features, dtype=bool)

    current_train_scores = np.zeros(len(y_train), dtype=np.float32)
    current_test_scores = np.zeros(len(y_test), dtype=np.float32)

    rows = []

    for k in range(1, max_additions + 1):
        remaining_indices = np.where(remaining_mask)[0]

        best_clean_idx, best_train_candidate_bal_acc = (
            find_best_next_neuron_by_train_balanced_accuracy(
                train_contrib=train_contrib,
                y_train=y_train,
                current_train_scores=current_train_scores,
                remaining_indices=remaining_indices,
                batch_size=CANDIDATE_BATCH_SIZE,
            )
        )

        selected.append(best_clean_idx)
        remaining_mask[best_clean_idx] = False

        current_train_scores += train_contrib[:, best_clean_idx]
        current_test_scores += test_contrib[:, best_clean_idx]

        train_stats = accuracy_stats(y_train, current_train_scores)
        test_stats = accuracy_stats(y_test, current_test_scores)

        original_idx = int(kept_original_neuron_indices[best_clean_idx])

        row = {
            "k": k,
            "added_clean_feature_index": int(best_clean_idx),
            "added_original_neuron_index": original_idx,

            # This is the value used for selection at this step.
            "selection_train_balanced_accuracy": float(best_train_candidate_bal_acc),

            # Metrics after adding the selected neuron.
            "train_accuracy": train_stats["accuracy"],
            "train_balanced_accuracy": train_stats["balanced_accuracy"],
            "train_auc": train_stats["auc"],

            "test_accuracy": test_stats["accuracy"],
            "test_balanced_accuracy": test_stats["balanced_accuracy"],
            "test_auc": test_stats["auc"],

            "selected_clean_feature_indices": ",".join(map(str, selected)),
            "selected_original_neuron_indices": ",".join(
                map(str, kept_original_neuron_indices[np.array(selected)])
            ),
        }

        rows.append(row)

        print(
            f"[GREEDY] k={k:5d}/{max_additions} "
            f"added_original={original_idx:6d} "
            f"train_bal_acc={train_stats['balanced_accuracy']:.4f} "
            f"test_acc={test_stats['accuracy']:.4f} "
            f"test_bal_acc={test_stats['balanced_accuracy']:.4f}"
        )

    curve = pd.DataFrame(rows)
    curve.to_csv(OUT_DIR / "greedy_neuron_addition_curve.csv", index=False)

    return curve


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    np.random.seed(RANDOM_SEED)

    print("=" * 80)
    print("Loading neural data")
    print("=" * 80)

    X_pres = load_neural_presentations()

    print("=" * 80)
    print("Averaging presentations by stimulus")
    print("=" * 80)

    X_avg, presentation_counts = average_presentations_by_stimulus(X_pres)

    print("=" * 80)
    print("Loading ViT logits and labels")
    print("=" * 80)

    vit_logits = load_vit_natural_scenes_logits()
    y = make_labels_from_vit_logits(vit_logits)

    if X_avg.shape[0] != len(y):
        raise ValueError(
            f"X has {X_avg.shape[0]} rows, but y has {len(y)} labels."
        )

    print("=" * 80)
    print("Making stratified 80/20 train/test split")
    print("=" * 80)

    X_train_raw, X_test_raw, y_train, y_test, train_idx, test_idx = (
        make_stratified_train_test_split(X_avg, y)
    )

    print("=" * 80)
    print("Cleaning features using train statistics")
    print("=" * 80)

    X_train, X_test, kept_original_neuron_indices = clean_features_using_train(
        X_train_raw,
        X_test_raw,
    )

    print("=" * 80)
    print("Running greedy neuron addition")
    print("=" * 80)

    curve = greedy_neuron_addition_curve(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        kept_original_neuron_indices=kept_original_neuron_indices,
        max_additions=MAX_ADDITIONS,
    )

    best_train_row = curve.loc[curve["train_balanced_accuracy"].idxmax()]
    best_test_row_for_reporting_only = curve.loc[curve["test_balanced_accuracy"].idxmax()]

    summary = {
        "neural_file": str(NEURAL_FILE),
        "vit_file": str(VIT_FILE),
        "vit_key": VIT_KEY,
        "n_stimuli": int(N_STIMULI),
        "presentation_level_shape": list(X_pres.shape),
        "stimulus_averaged_shape": list(X_avg.shape),
        "clean_train_shape": list(X_train.shape),
        "clean_test_shape": list(X_test.shape),
        "n_clean_neurons": int(X_train.shape[1]),
        "presentation_order_assumption": PRESENTATION_ORDER,
        "used_explicit_stimulus_ids": bool(STIMULUS_IDS_FILE.exists()),
        "min_presentations_per_stimulus": int(presentation_counts.min()),
        "max_presentations_per_stimulus": int(presentation_counts.max()),
        "class_counts_all": {
            "inanimate": int((y == 0).sum()),
            "animate": int((y == 1).sum()),
        },
        "class_counts_train": {
            "inanimate": int((y_train == 0).sum()),
            "animate": int((y_train == 1).sum()),
        },
        "class_counts_test": {
            "inanimate": int((y_test == 0).sum()),
            "animate": int((y_test == 1).sum()),
        },
        "test_fraction": 0.2,
        "max_additions": None if MAX_ADDITIONS is None else int(MAX_ADDITIONS),

        # This is the best k chosen by train performance.
        "best_train_selected_k": int(best_train_row["k"]),
        "best_train_selected_test_accuracy": float(best_train_row["test_accuracy"]),
        "best_train_selected_test_balanced_accuracy": float(
            best_train_row["test_balanced_accuracy"]
        ),
        "best_train_selected_test_auc": float(best_train_row["test_auc"]),

        # Reporting only: do not use this to choose a model.
        "best_test_k_reporting_only": int(best_test_row_for_reporting_only["k"]),
        "best_test_accuracy_reporting_only": float(
            best_test_row_for_reporting_only["test_accuracy"]
        ),
        "best_test_balanced_accuracy_reporting_only": float(
            best_test_row_for_reporting_only["test_balanced_accuracy"]
        ),
    }

    with open(OUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("=" * 80)
    print("Summary")
    print("=" * 80)
    print(json.dumps(summary, indent=2))

    print("=" * 80)
    print(f"Done. Results saved to: {OUT_DIR}")
    print("=" * 80)


if __name__ == "__main__":
    main()

Loading neural data
[INFO] Raw neural shape: (39209, 118)
[INFO] Interpreting raw neural matrix as neurons x presentations.
[INFO] Presentation-level neural shape: (118, 39209)
Averaging presentations by stimulus
[WARN] No explicit stimulus_ids.npy found. Assuming PRESENTATION_ORDER='block'. Repeats per stimulus=1.
[INFO] Averaged neural responses by stimulus.
[INFO] Stimulus-averaged neural shape: (118, 39209)
[INFO] Presentations per stimulus: min=1, max=1
Loading ViT logits and labels
[INFO] ViT logits shape: (118, 1000)
[INFO] Derived animate/inanimate labels from ViT top-1.
[INFO] Inanimate count: 55
[INFO] Animate count:   63
Making stratified 80/20 train/test split
[INFO] Stratified 80/20 train/test split:
  train: n= 94, inanimate= 44, animate= 50
  test : n= 24, inanimate= 11, animate= 13
Cleaning features using train statistics
[INFO] Clean train shape: (94, 39209)
[INFO] Clean test shape:  (24, 39209)
Running greedy neuron addition
[INFO] Computing train/test contribution ma

In [6]:
#!/usr/bin/env python3
"""
LOO greedy poor man's classifier using only 1000 candidate neurons per fold.

Purpose
-------
For each left-out stimulus:

    1. Use the other 117 stimuli as training data.

    2. Select a pool of 1000 candidate neurons using training data only.
       This avoids scanning all 39,209 neurons during every greedy step.

    3. Greedily add neurons from that 1000-neuron pool.
       At each step, select the neuron whose addition maximizes TRAIN balanced accuracy.

    4. Evaluate the selected subset on the single held-out stimulus.

After all 118 LOO folds, aggregate predictions by k:

    k = 1, 2, ..., MAX_ADDITIONS

and report:

    LOO accuracy
    LOO balanced accuracy
    LOO AUC

Important
---------
The held-out stimulus is NEVER used to choose:
    - the 1000 candidate neurons
    - the next greedy neuron
    - the classifier midpoint/direction

This is still not a perfectly nested estimate if you use the final LOO curve to pick k
and report that same best k as final performance. But it is much better for exploration
than a single 80/20 split.

Expected files
--------------
/home/maria/Science/data/
    hybrid_neural_responses_reduced.npy
    google_vit-base-patch16-224_embeddings_logits.pkl

Outputs
-------
/home/maria/Science/thesis/experiments/007--PoorMansClassifier/
    loo_1000_neuron_greedy_results/
        vit_derived_labels.csv
        loo_curve_by_k.csv
        loo_fold_predictions_by_k.csv
        loo_selected_neurons_by_fold.csv
        summary.json
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
)
from sklearn.model_selection import LeaveOneOut


# =============================================================================
# Config
# =============================================================================

BASE_DIR = Path("/home/maria/Science/thesis/experiments/007--PoorMansClassifier")
DATA_DIR = Path("/home/maria/Science/data")
OUT_DIR = BASE_DIR / "loo_1000_neuron_greedy_results"
OUT_DIR.mkdir(exist_ok=True, parents=True)

NEURAL_FILE = DATA_DIR / "hybrid_neural_responses_reduced.npy"
VIT_FILE = DATA_DIR / "google_vit-base-patch16-224_embeddings_logits.pkl"
VIT_KEY = "natural_scenes"

N_STIMULI = 118
RANDOM_SEED = 42
ANIMATE_TOP1_THRESHOLD = 397

# Candidate-pool size per LOO fold.
# This is what keeps the experiment sane.
CANDIDATE_POOL_SIZE = 1000

# Maximum greedy additions from the 1000-neuron pool.
# Since your useful k looked small-ish, start with 100.
# Increase to 200/300 if the LOO curve is still rising.
MAX_ADDITIONS = 100

# Batch size for scoring all candidate neurons.
BATCH_SIZE = 2048

# Stop within a fold when the best possible next addition does not improve
# training balanced accuracy by at least this much.
MIN_TRAIN_IMPROVEMENT = 1e-12


# =============================================================================
# Loading
# =============================================================================

def load_neural_stimulus_matrix() -> np.ndarray:
    """
    Load neural matrix and return:

        X: stimuli x neurons

    Handles either:

        (118, n_neurons)
        (n_neurons, 118)
    """
    if not NEURAL_FILE.exists():
        raise FileNotFoundError(f"Missing neural file: {NEURAL_FILE}")

    X_raw = np.asarray(np.load(NEURAL_FILE, allow_pickle=True))

    print(f"[INFO] Raw neural shape: {X_raw.shape}")

    if X_raw.ndim != 2:
        raise ValueError(f"Expected 2D neural matrix, got {X_raw.shape}")

    if X_raw.shape[0] == N_STIMULI:
        X = X_raw
        print("[INFO] Interpreting neural matrix as stimuli x neurons.")

    elif X_raw.shape[1] == N_STIMULI:
        X = X_raw.T
        print("[INFO] Interpreting neural matrix as neurons x stimuli; transposing.")

    else:
        raise ValueError(
            f"Could not infer orientation from neural shape {X_raw.shape}. "
            f"Expected one axis to equal {N_STIMULI}."
        )

    X = X.astype(np.float32, copy=False)

    print(f"[INFO] Stimulus-level neural shape: {X.shape}")

    return X


def load_vit_logits() -> np.ndarray:
    """
    Load ViT logits.

    Expected:

        np.load(VIT_FILE, allow_pickle=True)["natural_scenes"]
    """
    if not VIT_FILE.exists():
        raise FileNotFoundError(f"Missing ViT file: {VIT_FILE}")

    obj = np.load(VIT_FILE, allow_pickle=True)

    if hasattr(obj, "keys"):
        keys = list(obj.keys())

        if VIT_KEY not in keys:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in {VIT_FILE}. "
                f"Available keys: {keys}"
            )

        logits = np.asarray(obj[VIT_KEY])

    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        item = obj.item()

        if not isinstance(item, dict):
            raise TypeError(f"Expected object array containing dict, got {type(item)}")

        if VIT_KEY not in item:
            raise KeyError(
                f"Key {VIT_KEY!r} not found in object dict. "
                f"Available keys: {list(item.keys())}"
            )

        logits = np.asarray(item[VIT_KEY])

    else:
        raise TypeError(f"Unsupported ViT object type: {type(obj)}")

    if logits.ndim != 2:
        raise ValueError(f"Expected 2D ViT logits, got {logits.shape}")

    if logits.shape[0] != N_STIMULI:
        raise ValueError(
            f"Expected {N_STIMULI} ViT rows, got {logits.shape[0]}"
        )

    print(f"[INFO] ViT logits shape: {logits.shape}")

    return logits.astype(np.float32, copy=False)


def make_labels_from_vit_logits(logits: np.ndarray) -> np.ndarray:
    """
    Derive labels:

        1 = animate
        0 = inanimate

    Rule:

        top1 <= 397 => animate
    """
    top1 = np.argmax(logits, axis=1)
    y = (top1 <= ANIMATE_TOP1_THRESHOLD).astype(int)

    print("[INFO] Derived animate/inanimate labels from ViT top-1.")
    print(f"[INFO] Inanimate count: {int((y == 0).sum())}")
    print(f"[INFO] Animate count:   {int((y == 1).sum())}")

    pd.DataFrame(
        {
            "stimulus_index": np.arange(N_STIMULI),
            "top1_imagenet_class": top1,
            "label_animate": y,
        }
    ).to_csv(OUT_DIR / "vit_derived_labels.csv", index=False)

    return y


# =============================================================================
# Poor man's classifier utilities
# =============================================================================

def compute_mean_difference_contributions(
    X_fit: np.ndarray,
    y_fit: np.ndarray,
    X_eval: np.ndarray,
) -> np.ndarray:
    """
    Fit mean-difference decoder on X_fit/y_fit and return per-neuron contributions
    on X_eval.

    For neuron j:

        w_j = mu_animate_j - mu_inanimate_j
        midpoint_j = 0.5 * (mu_animate_j + mu_inanimate_j)

        contribution_ij = (x_ij - midpoint_j) * w_j

    Score for selected neuron set S:

        score_i = sum_j contribution_ij
    """
    mu0 = X_fit[y_fit == 0].mean(axis=0)
    mu1 = X_fit[y_fit == 1].mean(axis=0)

    w = mu1 - mu0
    midpoint = 0.5 * (mu1 + mu0)

    contrib = (X_eval - midpoint) * w

    return contrib.astype(np.float32, copy=False)


def scores_to_preds(scores: np.ndarray) -> np.ndarray:
    return (scores > 0).astype(int)


def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, scores))


def balanced_accuracy_for_candidate_matrix(
    y_true: np.ndarray,
    score_matrix: np.ndarray,
) -> np.ndarray:
    """
    Compute balanced accuracy for many candidate score vectors.

    score_matrix:

        n_samples x n_candidates
    """
    preds = score_matrix > 0
    y_bool = y_true.astype(bool)

    pos_mask = y_bool
    neg_mask = ~y_bool

    if pos_mask.sum() == 0 or neg_mask.sum() == 0:
        raise ValueError("Need both classes to compute balanced accuracy.")

    tpr = preds[pos_mask].mean(axis=0)
    tnr = (~preds[neg_mask]).mean(axis=0)

    return 0.5 * (tpr + tnr)


def single_neuron_train_balanced_accuracies(
    train_contrib: np.ndarray,
    y_train: np.ndarray,
    batch_size: int = BATCH_SIZE,
) -> np.ndarray:
    """
    Score each neuron alone on training data.

    Used only to select the 1000-neuron candidate pool.
    """
    n_features = train_contrib.shape[1]
    scores = np.empty(n_features, dtype=np.float32)

    zero_scores = np.zeros(len(y_train), dtype=np.float32)

    for start in range(0, n_features, batch_size):
        end = min(start + batch_size, n_features)
        batch_scores = zero_scores[:, None] + train_contrib[:, start:end]

        scores[start:end] = balanced_accuracy_for_candidate_matrix(
            y_train,
            batch_scores,
        )

    return scores


def select_candidate_pool_from_train(
    X_train: np.ndarray,
    y_train: np.ndarray,
    kept_original_indices: np.ndarray,
    pool_size: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Select top candidate neurons using training data only.

    Returns:
        candidate_clean_indices
        candidate_train_scores
    """
    train_contrib_all = compute_mean_difference_contributions(
        X_fit=X_train,
        y_fit=y_train,
        X_eval=X_train,
    )

    single_scores = single_neuron_train_balanced_accuracies(
        train_contrib=train_contrib_all,
        y_train=y_train,
    )

    # Tie-breaker:
    # If many neurons have identical balanced accuracy, prefer larger absolute
    # mean contribution. This avoids deterministic low-index tie goblins.
    abs_signal = np.mean(np.abs(train_contrib_all), axis=0)

    order = np.lexsort((-abs_signal, -single_scores))
    candidate_clean_indices = order[:pool_size]

    candidate_train_scores = single_scores[candidate_clean_indices]

    return candidate_clean_indices.astype(int), candidate_train_scores


def find_best_next_neuron(
    train_contrib_pool: np.ndarray,
    y_train: np.ndarray,
    current_train_scores: np.ndarray,
    remaining_mask: np.ndarray,
) -> tuple[int, float]:
    """
    Find best next neuron inside candidate pool.

    Returns:
        pool_index
        best_train_balanced_accuracy
    """
    remaining = np.where(remaining_mask)[0]

    best_pool_idx = None
    best_score = -np.inf

    for start in range(0, len(remaining), BATCH_SIZE):
        batch = remaining[start:start + BATCH_SIZE]

        candidate_score_matrix = (
            current_train_scores[:, None] + train_contrib_pool[:, batch]
        )

        batch_scores = balanced_accuracy_for_candidate_matrix(
            y_train,
            candidate_score_matrix,
        )

        local_pos = int(np.argmax(batch_scores))
        local_score = float(batch_scores[local_pos])

        if local_score > best_score:
            best_score = local_score
            best_pool_idx = int(batch[local_pos])

    assert best_pool_idx is not None

    return best_pool_idx, best_score


# =============================================================================
# LOO fold
# =============================================================================

def run_one_loo_fold(
    X: np.ndarray,
    y: np.ndarray,
    heldout_idx: int,
) -> tuple[list[dict], dict]:
    """
    Run one LOO fold.

    Returns:
        selected_rows
        prediction_info
    """
    train_idx = np.array([i for i in range(len(y)) if i != heldout_idx], dtype=int)
    test_idx = np.array([heldout_idx], dtype=int)

    X_train_raw = X[train_idx]
    y_train = y[train_idx]

    X_test_raw = X[test_idx]
    y_test = y[test_idx]

    # Clean based on this fold's train data.
    finite = np.isfinite(X_train_raw).all(axis=0) & np.isfinite(X_test_raw).all(axis=0)
    train_var = np.nanvar(X_train_raw, axis=0)
    nonzero = train_var > 0

    keep = finite & nonzero
    kept_original_indices = np.where(keep)[0]

    X_train = X_train_raw[:, keep].astype(np.float32, copy=False)
    X_test = X_test_raw[:, keep].astype(np.float32, copy=False)

    if X_train.shape[1] == 0:
        raise RuntimeError(f"No usable neurons in fold heldout_idx={heldout_idx}")

    pool_size = min(CANDIDATE_POOL_SIZE, X_train.shape[1])

    candidate_clean_indices, candidate_single_scores = select_candidate_pool_from_train(
        X_train=X_train,
        y_train=y_train,
        kept_original_indices=kept_original_indices,
        pool_size=pool_size,
    )

    candidate_original_indices = kept_original_indices[candidate_clean_indices]

    X_train_pool = X_train[:, candidate_clean_indices]
    X_test_pool = X_test[:, candidate_clean_indices]

    train_contrib_pool = compute_mean_difference_contributions(
        X_fit=X_train_pool,
        y_fit=y_train,
        X_eval=X_train_pool,
    )

    test_contrib_pool = compute_mean_difference_contributions(
        X_fit=X_train_pool,
        y_fit=y_train,
        X_eval=X_test_pool,
    )

    max_additions = min(MAX_ADDITIONS, pool_size)

    current_train_scores = np.zeros(len(y_train), dtype=np.float32)
    current_test_score = np.float32(0.0)

    current_train_bal_acc = 0.5

    remaining_mask = np.ones(pool_size, dtype=bool)
    selected_pool_indices: list[int] = []

    selected_rows: list[dict] = []

    # Store held-out score/pred for each k.
    heldout_scores_by_k = np.zeros(MAX_ADDITIONS + 1, dtype=np.float32)
    heldout_preds_by_k = np.zeros(MAX_ADDITIONS + 1, dtype=int)

    # k = 0 baseline: score 0 => predict 0.
    heldout_scores_by_k[0] = float(current_test_score)
    heldout_preds_by_k[0] = int(current_test_score > 0)

    stopped_at = 0

    for k in range(1, max_additions + 1):
        best_pool_idx, best_new_train_bal_acc = find_best_next_neuron(
            train_contrib_pool=train_contrib_pool,
            y_train=y_train,
            current_train_scores=current_train_scores,
            remaining_mask=remaining_mask,
        )

        improvement = best_new_train_bal_acc - current_train_bal_acc

        if improvement < MIN_TRAIN_IMPROVEMENT:
            stopped_at = k - 1
            print(
                f"[LOO {heldout_idx:03d}] stop at k={stopped_at}; "
                f"no train improvement. best_possible={best_new_train_bal_acc:.4f}, "
                f"current={current_train_bal_acc:.4f}"
            )
            break

        selected_pool_indices.append(best_pool_idx)
        remaining_mask[best_pool_idx] = False

        current_train_scores += train_contrib_pool[:, best_pool_idx]
        current_test_score += test_contrib_pool[0, best_pool_idx]

        current_train_bal_acc = float(
            balanced_accuracy_score(y_train, scores_to_preds(current_train_scores))
        )
        current_train_acc = float(
            accuracy_score(y_train, scores_to_preds(current_train_scores))
        )

        heldout_score = float(current_test_score)
        heldout_pred = int(heldout_score > 0)

        heldout_scores_by_k[k] = heldout_score
        heldout_preds_by_k[k] = heldout_pred

        selected_rows.append(
            {
                "heldout_stimulus_index": int(heldout_idx),
                "heldout_true_label": int(y_test[0]),
                "k": int(k),
                "added_pool_index": int(best_pool_idx),
                "added_clean_feature_index": int(candidate_clean_indices[best_pool_idx]),
                "added_original_neuron_index": int(candidate_original_indices[best_pool_idx]),
                "train_accuracy": current_train_acc,
                "train_balanced_accuracy": current_train_bal_acc,
                "heldout_score": heldout_score,
                "heldout_pred": heldout_pred,
                "heldout_correct": int(heldout_pred == int(y_test[0])),
                "candidate_single_train_balanced_accuracy": float(
                    candidate_single_scores[best_pool_idx]
                ),
            }
        )

        print(
            f"[LOO {heldout_idx:03d}] k={k:3d}/{max_additions} "
            f"add_orig={int(candidate_original_indices[best_pool_idx]):6d} "
            f"train_bal_acc={current_train_bal_acc:.4f} "
            f"heldout_true={int(y_test[0])} pred={heldout_pred} "
            f"score={heldout_score:+.4f}"
        )

        stopped_at = k

    # If fold stopped early, fill later k with final unchanged prediction.
    for k in range(stopped_at + 1, MAX_ADDITIONS + 1):
        heldout_scores_by_k[k] = heldout_scores_by_k[stopped_at]
        heldout_preds_by_k[k] = heldout_preds_by_k[stopped_at]

    prediction_info = {
        "heldout_stimulus_index": int(heldout_idx),
        "true_label": int(y_test[0]),
        "stopped_at_k": int(stopped_at),
        "scores_by_k": heldout_scores_by_k.tolist(),
        "preds_by_k": heldout_preds_by_k.tolist(),
    }

    return selected_rows, prediction_info


# =============================================================================
# Aggregate
# =============================================================================

def aggregate_loo_predictions(prediction_infos: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build:
        1. loo_fold_predictions_by_k.csv
        2. loo_curve_by_k.csv
    """
    pred_rows = []

    for info in prediction_infos:
        heldout_idx = info["heldout_stimulus_index"]
        true_label = info["true_label"]
        stopped_at = info["stopped_at_k"]

        scores_by_k = info["scores_by_k"]
        preds_by_k = info["preds_by_k"]

        for k in range(1, MAX_ADDITIONS + 1):
            pred = int(preds_by_k[k])
            score = float(scores_by_k[k])

            pred_rows.append(
                {
                    "heldout_stimulus_index": heldout_idx,
                    "k": k,
                    "true_label": true_label,
                    "pred": pred,
                    "score": score,
                    "correct": int(pred == true_label),
                    "fold_stopped_at_k": stopped_at,
                }
            )

    pred_df = pd.DataFrame(pred_rows)

    curve_rows = []

    for k, group in pred_df.groupby("k"):
        y_true = group["true_label"].to_numpy()
        y_pred = group["pred"].to_numpy()
        scores = group["score"].to_numpy()

        curve_rows.append(
            {
                "k": int(k),
                "loo_accuracy": float(accuracy_score(y_true, y_pred)),
                "loo_balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
                "loo_auc": safe_auc(y_true, scores),
                "n_correct": int((y_true == y_pred).sum()),
                "n_total": int(len(y_true)),
            }
        )

    curve_df = pd.DataFrame(curve_rows).sort_values("k")

    pred_df.to_csv(OUT_DIR / "loo_fold_predictions_by_k.csv", index=False)
    curve_df.to_csv(OUT_DIR / "loo_curve_by_k.csv", index=False)

    return pred_df, curve_df


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    np.random.seed(RANDOM_SEED)

    print("=" * 80)
    print("Loading data")
    print("=" * 80)

    X = load_neural_stimulus_matrix()
    logits = load_vit_logits()
    y = make_labels_from_vit_logits(logits)

    if X.shape[0] != len(y):
        raise ValueError(f"X has {X.shape[0]} rows but y has {len(y)} labels.")

    print("=" * 80)
    print("Running LOO greedy experiment")
    print("=" * 80)

    loo = LeaveOneOut()

    all_selected_rows: list[dict] = []
    prediction_infos: list[dict] = []

    for fold_num, (_, test_idx) in enumerate(loo.split(X), start=1):
        heldout_idx = int(test_idx[0])

        print("=" * 80)
        print(f"LOO fold {fold_num}/{N_STIMULI}: heldout stimulus {heldout_idx}")
        print("=" * 80)

        selected_rows, prediction_info = run_one_loo_fold(
            X=X,
            y=y,
            heldout_idx=heldout_idx,
        )

        all_selected_rows.extend(selected_rows)
        prediction_infos.append(prediction_info)

    selected_df = pd.DataFrame(all_selected_rows)
    selected_df.to_csv(OUT_DIR / "loo_selected_neurons_by_fold.csv", index=False)

    pred_df, curve_df = aggregate_loo_predictions(prediction_infos)

    best_bal_row = curve_df.loc[curve_df["loo_balanced_accuracy"].idxmax()]
    best_acc_row = curve_df.loc[curve_df["loo_accuracy"].idxmax()]

    stopped_ks = np.array([info["stopped_at_k"] for info in prediction_infos])

    summary = {
        "neural_file": str(NEURAL_FILE),
        "vit_file": str(VIT_FILE),
        "vit_key": VIT_KEY,
        "X_shape": list(X.shape),
        "n_stimuli": int(N_STIMULI),
        "n_neurons_total": int(X.shape[1]),
        "candidate_pool_size": int(CANDIDATE_POOL_SIZE),
        "max_additions": int(MAX_ADDITIONS),
        "class_counts": {
            "inanimate": int((y == 0).sum()),
            "animate": int((y == 1).sum()),
        },
        "mean_fold_stopped_at_k": float(stopped_ks.mean()),
        "median_fold_stopped_at_k": float(np.median(stopped_ks)),
        "min_fold_stopped_at_k": int(stopped_ks.min()),
        "max_fold_stopped_at_k": int(stopped_ks.max()),

        # Reporting only: if you choose k by this same LOO curve,
        # the estimate is optimistic for final model selection.
        "best_loo_balanced_accuracy_k_reporting_only": int(best_bal_row["k"]),
        "best_loo_balanced_accuracy_reporting_only": float(
            best_bal_row["loo_balanced_accuracy"]
        ),
        "best_loo_accuracy_at_best_balanced_k_reporting_only": float(
            best_bal_row["loo_accuracy"]
        ),
        "best_loo_auc_at_best_balanced_k_reporting_only": float(
            best_bal_row["loo_auc"]
        ),
        "best_loo_accuracy_k_reporting_only": int(best_acc_row["k"]),
        "best_loo_accuracy_reporting_only": float(best_acc_row["loo_accuracy"]),
        "best_loo_balanced_accuracy_at_best_accuracy_k_reporting_only": float(
            best_acc_row["loo_balanced_accuracy"]
        ),
    }

    with open(OUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("=" * 80)
    print("Summary")
    print("=" * 80)
    print(json.dumps(summary, indent=2))

    print("=" * 80)
    print(f"Done. Results saved to: {OUT_DIR}")
    print("=" * 80)


if __name__ == "__main__":
    main()

Loading data
[INFO] Raw neural shape: (39209, 118)
[INFO] Interpreting neural matrix as neurons x stimuli; transposing.
[INFO] Stimulus-level neural shape: (118, 39209)
[INFO] ViT logits shape: (118, 1000)
[INFO] Derived animate/inanimate labels from ViT top-1.
[INFO] Inanimate count: 55
[INFO] Animate count:   63
Running LOO greedy experiment
LOO fold 1/118: heldout stimulus 0
[LOO 000] k=  1/100 add_orig= 35669 train_bal_acc=0.7449 heldout_true=1 pred=1 score=+0.0011
[LOO 000] k=  2/100 add_orig= 29128 train_bal_acc=0.8013 heldout_true=1 pred=0 score=-0.0001
[LOO 000] k=  3/100 add_orig=  6365 train_bal_acc=0.8367 heldout_true=1 pred=0 score=-0.0001
[LOO 000] k=  4/100 add_orig= 29382 train_bal_acc=0.8619 heldout_true=1 pred=0 score=-0.0001
[LOO 000] k=  5/100 add_orig= 19815 train_bal_acc=0.8699 heldout_true=1 pred=0 score=-0.0003
[LOO 000] k=  6/100 add_orig= 38926 train_bal_acc=0.8780 heldout_true=1 pred=0 score=-0.0008
[LOO 000] k=  7/100 add_orig= 28932 train_bal_acc=0.8861 held